### Step 0: Cohort Data Ingestion & Tensor Assembly

Load and aggregate preprocessed 3D epoch arrays, target labels, and metadata across all 102 benchmark subjects ($X \in \mathbb{R}^{N_{\text{total\_trials}} \times 64 \times 641}$, $y \in \{0, 1\}^{N_{\text{total\_trials}}}$).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import mne

# Add source directory to Python path
# sys.path.append(os.path.abspath('../srcs'))
from misc import load_and_parse_eeg

# Suppress verbose MNE logs
mne.set_log_level('WARNING')

# 1. Define Dataset Path & Benchmark Cohort (102 Clean Subjects)
BASE_DATA_PATH = "mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"
ANOMALY_EXCLUDED_SUBJECTS = [38, 88, 89, 92, 100, 104, 106]
CLEAN_SUBJECTS = [s for s in range(1, 110) if s not in ANOMALY_EXCLUDED_SUBJECTS]
RUN_IDS = [4, 8, 12]  # Unilateral Motor Imagery: Left vs. Right Fist


In [2]:


print(f"Cohort Configuration:")
print(f"  Total Subjects Requested : {len(CLEAN_SUBJECTS)}")
print(f"  Excluded Anomaly Subjects: {ANOMALY_EXCLUDED_SUBJECTS}")
print(f"  Target Run IDs           : {RUN_IDS}\n")



Cohort Configuration:
  Total Subjects Requested : 102
  Excluded Anomaly Subjects: [38, 88, 89, 92, 100, 104, 106]
  Target Run IDs           : [4, 8, 12]



In [3]:


# 2. Ingest, Filter, and Epoch Cohort Data into 3D Tensor
X, y, df_metadata = load_and_parse_eeg(
    subject_ids=CLEAN_SUBJECTS,
    run_ids=RUN_IDS,
    base_path=BASE_DATA_PATH,
    tmin=0.0,
    tmax=4.0,
    include_rest=False
)

Processing Subjects:   0%|          | 0/102 [00:00<?, ?it/s]

Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R08.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 montage applied.
Loading data from mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R12.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1020 mon

In [4]:
# 3. Cohort-Wide Integrity & Dimensional Verification
unique_subjects = np.unique(df_metadata['subject_id'])
unique_subjects


array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  39,  40,
        41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
        54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,
        67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,
        80,  81,  82,  83,  84,  85,  86,  87,  90,  91,  93,  94,  95,
        96,  97,  98,  99, 101, 102, 103, 105, 107, 108, 109])

In [5]:

class_counts = np.bincount(y)
class_counts

array([2304, 2267])

In [6]:

print("--- Cohort Data Ingestion Integrity Summary ---")
print(f"Epoch Tensor (X) Shape    : {X.shape} (dtype: {X.dtype})")
print(f"Target Vector (y) Shape   : {y.shape} (dtype: {y.dtype})")
print(f"Metadata Records Shape    : {df_metadata.shape}")
print(f"Unique Subjects Loaded    : {len(unique_subjects)} / 102 (Target: 102)")
print(f"Class Balance (0 vs 1)    : Class 0 = {class_counts[0]}, Class 1 = {class_counts[1]} (Ratio: {class_counts[0]/class_counts[1]:.2f})")
print(f"Signal Amplitude (uV/V)   : Mean = {X.mean():.4e}, Std = {X.std():.4e}, Min = {X.min():.4e}, Max = {X.max():.4e}")

--- Cohort Data Ingestion Integrity Summary ---
Epoch Tensor (X) Shape    : (4571, 64, 641) (dtype: float64)
Target Vector (y) Shape   : (4571,) (dtype: int64)
Metadata Records Shape    : (4571, 5)
Unique Subjects Loaded    : 102 / 102 (Target: 102)
Class Balance (0 vs 1)    : Class 0 = 2304, Class 1 = 2267 (Ratio: 1.02)
Signal Amplitude (uV/V)   : Mean = -5.9437e-25, Std = 1.4706e-05, Min = -1.0648e-03, Max = 1.0420e-03


In [7]:

# Assert cohort integrity constraints
assert len(unique_subjects) == 102, f"Expected 102 unique subjects, got {len(unique_subjects)}"
assert X.ndim == 3 and X.shape[1] == 64 and X.shape[2] == 641, f"Unexpected tensor shape: {X.shape}"
assert set(np.unique(y)) == {0, 1}, f"Unexpected target classes: {np.unique(y)}"
print("\n[PASSED] Step 0 Cohort Data Ingestion and Tensor Assembly successfully verified.")


[PASSED] Step 0 Cohort Data Ingestion and Tensor Assembly successfully verified.


## Comparison of ``Log Variance Transformation (naive baseline)`` vs. ``MNE CSP (spatial filtering)`` for dimensionality reduction and feature extraction.``

### Step 1: Cell 2.1 - Simple Feature Extractor Transformer (Non-CSP Baseline)

Define `LogVarTransformer` ($\mathbb{R}^{N_{\text{total\_trials}} \times 64 \times 641} \to \mathbb{R}^{N_{\text{total\_trials}} \times 64}$) computing signal power as $\log(\text{Var}(X, \text{axis}=2) + \epsilon)$.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogVarTransformer(BaseEstimator, TransformerMixin):
    """
    Computes log-variance of EEG signals along the time axis (axis=2).
    Target: Reduce 3D trial tensors to 2D feature matrices.
    """
    def __init__(self, epsilon=1e-10):
        self.epsilon = epsilon

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Calculate variance across the time dimension (axis=2)
        # Input: (N_trials, 64_channels, 641_samples) -> Output: (N_trials, 64_channels)
        # for more information on the log-variance transformation, see: https://app.notion.com/p/jvalenci/total-perspective-vortex-3929d52658e080088fdcc42b60719b01?source=copy_link#3e69d52658e08004ac60e8196034496a
        return np.log(np.var(X, axis=2) + self.epsilon)

In [11]:

# 1. Instantiate and Execute Baseline Extraction
logvar_extractor = LogVarTransformer()
logvar_extractor

,epsilon,1e-10


In [13]:

X_base = logvar_extractor.fit_transform(X)
X_base


array([[-22.17732316, -22.36892067, -22.35207147, ..., -21.82920396,
        -21.60198611, -21.56107417],
       [-22.30188671, -22.43594735, -22.42085517, ..., -21.32537197,
        -21.19701876, -21.16965324],
       [-22.12089295, -22.20380197, -22.2102903 , ..., -21.58022987,
        -21.20640607, -21.33379627],
       ...,
       [-21.73558929, -22.46825552, -21.33082997, ..., -21.82224172,
        -21.79481968, -21.9676679 ],
       [-21.89235882, -22.47787378, -21.17214557, ..., -21.93501519,
        -21.90752815, -22.09086619],
       [-21.64652395, -22.40569676, -21.23352864, ..., -21.77269608,
        -21.84519996, -21.93569093]], shape=(4571, 64))

In [14]:


# 2. Dimensional & Statistical Verification
print("--- Step 1: Log-Variance Feature Extraction (Non-CSP) ---")
print(f"Input Tensor Shape   : {X.shape}")
print(f"Output Matrix Shape  : {X_base.shape} (Target: {X.shape[0]}, 64)")
print(f"Feature Vector Dtype : {X_base.dtype}")
print(f"Cohort Stats (Mean)  : {X_base.mean():.4f}")
print(f"Cohort Stats (Std)   : {X_base.std():.4f}")

# Assert correct reduction to (N_trials, 64)
assert X_base.shape == (X.shape[0], 64), f"Dimensionality reduction failed: {X_base.shape}"
print("\n[PASSED] LogVarTransformer successfully reduced 3D cohort tensor to 2D feature matrix.")

--- Step 1: Log-Variance Feature Extraction (Non-CSP) ---
Input Tensor Shape   : (4571, 64, 641)
Output Matrix Shape  : (4571, 64) (Target: 4571, 64)
Feature Vector Dtype : float64
Cohort Stats (Mean)  : -22.2058
Cohort Stats (Std)   : 0.6613

[PASSED] LogVarTransformer successfully reduced 3D cohort tensor to 2D feature matrix.


### Step 2: Cell 2.2 - Standard MNE CSP Integration

Implement `mne.decoding.CSP` ($\mathbb{R}^{N_{\text{total\_trials}} \times 64 \times 641} \to \mathbb{R}^{N_{\text{total\_trials}} \times 6}$) to extract spatially filtered log-variance features across cohort trials.

In [15]:
from mne.decoding import CSP

# 1. Instantiate CSP with 6 components and Ledoit-Wolf regularization
csp = CSP(n_components=6, reg='ledoit_wolf', log=True, norm_trace=False)
csp

,n_components,6
,reg,'ledoit_wolf'
,log,True
,cov_est,'concat'
,transform_into,'average_power'
,norm_trace,False
,cov_method_params,None
,restr_type,'restricting'
,info,None
,rank,None
,component_order,'mutual_info'


In [16]:


# 2. Fit and Transform the Cohort Data
# Input: (N_trials, 64, 641) -> Output: (N_trials, 6)
X_csp = csp.fit_transform(X, y)
X_csp


array([[-2.17803022, -1.20355427, -1.23185528, -1.64894359, -1.98004471,
        -1.31816856],
       [-2.2653922 , -1.37903867, -1.33636749, -2.00417749, -2.41364567,
        -1.48799938],
       [-2.53757999, -1.28713872, -1.10564478, -1.48847235, -2.33905549,
        -0.86717253],
       ...,
       [-1.29424841, -0.85338021, -0.80265206, -1.34579751, -2.29443663,
        -0.01787066],
       [-1.74525286, -0.53568558, -0.61959539, -1.41222289, -2.16650837,
        -0.2095852 ],
       [-1.50693964, -0.629886  , -0.64472455, -1.37489842, -2.16434195,
        -0.02908909]], shape=(4571, 6))

In [17]:


# 3. Dimensional & Statistical Verification
print("--- Step 2: MNE Common Spatial Patterns (CSP) Extraction ---")
print(f"Input Tensor Shape   : {X.shape}")
print(f"Output Matrix Shape  : {X_csp.shape} (Target: {X.shape[0]}, 6)")
print(f"Feature Vector Dtype : {X_csp.dtype}")
print(f"CSP Feature Stats    : Min = {X_csp.min():.4f}, Max = {X_csp.max():.4f}, Mean = {X_csp.mean():.4f}")


--- Step 2: MNE Common Spatial Patterns (CSP) Extraction ---
Input Tensor Shape   : (4571, 64, 641)
Output Matrix Shape  : (4571, 6) (Target: 4571, 6)
Feature Vector Dtype : float64
CSP Feature Stats    : Min = -6.9374, Max = 4.6153, Mean = -1.7957


In [18]:


# Assert correct reduction to (N_trials, 6)
assert X_csp.shape == (X.shape[0], 6), f"CSP Dimensionality reduction failed: {X_csp.shape}"
print("\n[PASSED] MNE CSP successfully reduced 64-channel cohort data to 6 spatial components.")


[PASSED] MNE CSP successfully reduced 64-channel cohort data to 6 spatial components.


### Step 3: Cell 2.3 - Baseline Scikit-Learn Pipelines & Full-Cohort Evaluation

Construct Scikit-Learn pipelines using `LinearDiscriminantAnalysis` and evaluate mean accuracy across all 102 benchmark subjects targeting $\ge 60\%$.

In [19]:
# sklearn pipeline is a structured way to chain the transformations and model fitting steps together,
# allowing for cleaner code and easier cross-validation.
from sklearn.pipeline import Pipeline

# !The linear discriminant analysis (LDA) classifier is a simple yet effective
# * modele for binary classification tasks, especially in EEG signal analysis.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# * cross validation utilities for model evaluation( data splitting, scoring, etc.)
# * StratifiedKFold ensures that each fold of the cross-validation maintains the same class distribution
# as the original dataset, which is crucial for imbalanced datasets.
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1. Construct Baseline Pipelines


In [21]:

# !baseline pipeline using log-variance transformation followed by LDA without CSP
pipe_logvar = Pipeline(
    [("power", LogVarTransformer()), ("lda", LinearDiscriminantAnalysis())]
)

# !CSP pipeline using MNE CSP followed by LDA
pipe_csp = Pipeline(
    [
        ("csp", CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis()),
    ]
)

In [ ]:
# Data splitting this retuns 5 train-test splits, ensuring that each split maintains 
    # the same class distribution as the original dataset.
# 2. Pooled Cohort Evaluation (5-Fold Stratified Cross-Validation)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [24]:

print("--- Step 3: Full-Cohort Pipeline Evaluation ---")

# Evaluate Log-Var Pipeline
# Launch cross-validation for the baseline pipeline (Log-Variance + LDA) and compute accuracy scores.
scores_base = cross_val_score(pipe_logvar, X, y, cv=cv)
print(
    f"Cohort Log-Var + LDA Accuracy : {scores_base.mean():.2%} (+/- {scores_base.std():.2%})"
)

# Evaluate CSP Pipeline
# Launch cross-validation for the CSP pipeline (CSP + LDA) and compute accuracy scores.
scores_csp = cross_val_score(pipe_csp, X, y, cv=cv)
print(
    f"Cohort CSP + LDA Accuracy    : {scores_csp.mean():.2%} (+/- {scores_csp.std():.2%})"
)


--- Step 3: Full-Cohort Pipeline Evaluation ---
Cohort Log-Var + LDA Accuracy : 60.42% (+/- 1.12%)
Cohort CSP + LDA Accuracy    : 60.23% (+/- 2.29%)


In [25]:

# 3. Per-Subject Accuracy Distribution
subject_ids = np.unique(df_metadata["subject_id"])
subject_accuracies = []

for subj in subject_ids:
    mask = df_metadata["subject_id"] == subj
    # Calculate subject-specific mean CV accuracy
    # (Using 3-fold for individual subjects due to fewer trials per subject)
    s_score = cross_val_score(pipe_csp, X[mask], y[mask], cv=3).mean()
    subject_accuracies.append(s_score)

print(
    f"Mean Per-Subject Accuracy    : {np.mean(subject_accuracies):.2%} (+/- {np.std(subject_accuracies):.2%})"
)

Mean Per-Subject Accuracy    : 63.32% (+/- 12.51%)


In [26]:


# 4. Benchmark Threshold Verification (Target >= 60%)
mean_acc = scores_csp.mean()
target_met = mean_acc >= 0.60
status = "PASSED" if target_met else "FAILED"

print(f"\n[{status}] Overall Cohort Mean Accuracy: {mean_acc:.2%}")
print(f"Target Benchmark: >= 60.00%")

if target_met:
    print(
        "Requirement satisfied. Baseline established for Phase 3 algorithm development."
    )
else:
    print(
        "Warning: Baseline did not meet 60% requirement. Check filter parameters or artifacts."
    )


[PASSED] Overall Cohort Mean Accuracy: 60.23%
Target Benchmark: >= 60.00%
Requirement satisfied. Baseline established for Phase 3 algorithm development.
